In [1]:
import re 
import torch
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any 
from datasets import load_dataset


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"device: {device}")

device: mps


## Warmup

In [3]:
test = "test1 \n test2\\n \n test3"

print(test)

print(re.search(r"(\n)", test))
print(re.search("(\n)", test))
print(re.search("(\\n)", test))
print(re.search(r"(\\n)", test))
print(re.search("(\\\\n)", test))



test1 
 test2\n 
 test3
<re.Match object; span=(6, 7), match='\n'>
<re.Match object; span=(6, 7), match='\n'>
<re.Match object; span=(6, 7), match='\n'>
<re.Match object; span=(13, 15), match='\\n'>
<re.Match object; span=(13, 15), match='\\n'>


In [4]:
print(re.search(r"^[A-Z]", "The start."))
print(re.search(r".$", "The End?"))
print(re.search(r".$", "The end!"))
print(re.search(r"\.$", "The end."))


<re.Match object; span=(0, 1), match='T'>
<re.Match object; span=(7, 8), match='?'>
<re.Match object; span=(7, 8), match='!'>
<re.Match object; span=(7, 8), match='.'>


In [5]:
re.search(r"^[A-Z]", "The start.")

<re.Match object; span=(0, 1), match='T'>

In [6]:
pattern = f"[mM]"
text = "Mary Ann stopped by the market to buy some milk"
print(f"text: {text}")
first_match = re.search(pattern, text)
all_matches = re.findall(pattern, text)
all_matches_iter = re.finditer(pattern, text)

print(f"The first match is:{first_match}")
print(f"All matches are: {all_matches}")
print(f"All matches as an iterator: {all_matches_iter}")


for cnt, match in enumerate(all_matches_iter):
    print(
        f"match {cnt}: starts at {match.start()} and ends at {match.end()}"
    )

text: Mary Ann stopped by the market to buy some milk
The first match is:<re.Match object; span=(0, 1), match='M'>
All matches are: ['M', 'm', 'm', 'm']
All matches as an iterator: <callable_iterator object at 0x11a6bcd60>
match 0: starts at 0 and ends at 1
match 1: starts at 24 and ends at 25
match 2: starts at 40 and ends at 41
match 3: starts at 43 and ends at 44


### Nagation 

In [7]:
text2 = "Nagation with Capital N"
pattern2 = f"[^nN]"
print(f"text: {text2}")

result2 = re.search(pattern2, text2)
print(f"The first match is: {result2}")




text: Nagation with Capital N
The first match is: <re.Match object; span=(1, 2), match='a'>


### Digits and grouping

In [8]:
text3 = "This is a text containing digits:\n \
    1without whitespace, and 2, 33 with whitespaces."
print(f"text: {text3}")

pattern3 = r"[0-9]"  # \d 
first_digit = re.search(pattern3, text3)
print(f"The first match is: {first_digit}")
all_digits_indiv = re.findall(pattern3, text3)
print(f"All digits individually: {all_digits_indiv}")

pattern4 = r"([0-9]+)" 
all_digits_grouped = re.findall(pattern4, text3)
print(f"All digits grouped: {all_digits_grouped}")

# pattern5 = r"\b\w+\b"
# all_digits_with_WS = re.findall(pattern5, text3)
# print(f"All digits grouped with proper whitespace : {all_digits_grouped}")





text: This is a text containing digits:
     1without whitespace, and 2, 33 with whitespaces.
The first match is: <re.Match object; span=(39, 40), match='1'>
All digits individually: ['1', '2', '3', '3']
All digits grouped: ['1', '2', '33']


In [9]:
all_words_with_tT = re.findall(r"\b[tT]\w+", text3)
print(f"All words with tT: {all_words_with_tT}")

All words with tT: ['This', 'text']


### Grouping and subsitution 

In [10]:
text4 = "This is a text to show groupings and replacements \n \
    with US date format MM/DD/YYYY (12/06/1985) to EUR \n \
    format DD-MM-YYYY (06-12-1986). \n"
print(text4)
pattern5 = r"(\d{2})/(\d{2})/(\d{4})"
us_date = re.search(pattern5, text4)
print(f"US date: {us_date.group()} \n")
eur_date = re.sub(pattern5, r"\2-\1-3", text4)
print(f"Text replaced with EUR data: \n {eur_date}")


This is a text to show groupings and replacements 
     with US date format MM/DD/YYYY (12/06/1985) to EUR 
     format DD-MM-YYYY (06-12-1986). 

US date: 12/06/1985 

Text replaced with EUR data: 
 This is a text to show groupings and replacements 
     with US date format MM/DD/YYYY (06-12-3) to EUR 
     format DD-MM-YYYY (06-12-1986). 



### Non-capturing group and Lookahead 

(?: pattern) 

(?= pattern) 

(?! pattern)



In [11]:
text6 = "This is to demonstate an simple example of Non-capturing group: \n \
     by detecting the 4th digit of the following number 12 34 56 78 9"
print(text6)

pattern6 = r"(?:\d{2}\s+){4}(\d+)"

result6 = re.search(pattern6, text6)
print(f"The 4th digit is: {result6.group(1)}")

This is to demonstate an simple example of Non-capturing group: 
      by detecting the 4th digit of the following number 12 34 56 78 9
The 4th digit is: 9


In [12]:
text7 = "Let us detect the first word in the line with does not start with a L or l"

print(text7)

pattern7 = r"\b(?![lL])(\w+)\b"

result7 = re.search(pattern7, text7)
print(f"The detected digit is: {result7}")

Let us detect the first word in the line with does not start with a L or l
The detected digit is: <re.Match object; span=(4, 6), match='us'>


In [13]:
text = "Oh, We're 350 dogs! Um, lunch?" 

# token_pattern = r"\w+\s|\w+'\w+|\w+\W"
token_pattern = r"\w+(?:'\w+)?|[^\w\s]"
pretokenized = re.findall(token_pattern, text)
print(f"Pretokenized: {pretokenized}")


Pretokenized: ['Oh', ',', "We're", '350', 'dogs', '!', 'Um', ',', 'lunch', '?']
